### Sentiment Analysis for IMDB dataset

In [66]:
import numpy as np
import pandas as pd

In [67]:
import torch 
import torch.nn as nn 
import torch.optim as optim

In [68]:
df=pd.read_csv("IMDB Dataset.csv")

In [69]:
df.shape

(50000, 2)

In [70]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [71]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [72]:
df.drop_duplicates(inplace=True)

In [73]:
df.dropna()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [74]:
df.shape

(49582, 2)

### Preprocessing

In [75]:
# 1. convert to lowercase

df['review']=df['review'].str.lower()

In [76]:
# 2. Removing the URL's

import re 

In [77]:
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text

df['review']=df['review'].apply(remove_urls)

In [78]:

# 3. remove puntuations

def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df['review']=df['review'].apply(remove_punctuations)



In [79]:
# 4. removing html tags

def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

df['review']=df['review'].apply(remove_html)

In [80]:
# 5. removing stopwords

import nltk

In [81]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [82]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [83]:
sample_texts="I love coding in python"

tokens=word_tokenize(sample_texts)

print(tokens)

['I', 'love', 'coding', 'in', 'python']


In [84]:
def remove_stopwords(text):
    tokens=word_tokenize(text)

    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")

    return text

df['review']=df['review'].apply(remove_stopwords)


In [85]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [86]:
# 6. Stemming 

from nltk.stem import PorterStemmer

In [87]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)

    for token in tokens:
        stemmed_tokens=ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df['review']=df['review'].apply(stemming)

In [88]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [89]:
# 7. Encoding 

from sklearn.preprocessing import LabelEncoder 

le=LabelEncoder()

y=le.fit_transform(df['sentiment'])

In [91]:
y

array([1, 1, 1, ..., 0, 0, 0], shape=(49582,))

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [93]:
tf=TfidfVectorizer(max_features=5000)

In [94]:
X=tf.fit_transform(df['review'])

In [96]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2868)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4369)	0.041994187696759305
  (0, 4171)	0.17799685402440263
  (0, 3693)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3805)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4497)	0.07614066339174266
  (0, 3857)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

### Dataset and DataLoader 

In [114]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [115]:
X_train.shape

(39665, 5000)

In [116]:
X_test.shape

(9917, 5000)

In [117]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [118]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [120]:
trainset=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float()
)

testset=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float()
)

In [121]:
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)
testloader=DataLoader(testset,batch_size=32,shuffle=True)

### Build the RNN

In [125]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super(RNN,self).__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn=nn.RNN(
            input_size,hidden_size,num_layers,batch_first=True
        )

        self.fc_layer=nn.Linear(
            hidden_size,1
        )

    def forward(self,x):
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_=self.rnn(x,h0)

        output=self.fc_layer(out[:,-1,:])

        return output

In [127]:
input_size=X_train.shape[1]
model=RNN(input_size)

In [128]:
criteria=nn.BCELoss()

In [129]:
optimizer=optim.Adam(model.parameters())

### Training the RNN

In [131]:
epochs=10

for epoch in range(epochs):

    model.train()

    for Xb,yb in trainloader:

        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1)

        outputs=model(Xb)

        outputs=outputs.squeeze()

        outputs=torch.sigmoid(outputs)

        loss=criteria(outputs,yb)

        loss.backward()

        optimizer.step()

    print(f"{epoch+1}/{epochs} => loss is {loss.item()}")



1/10 => loss is 0.45432791113853455
2/10 => loss is 0.27469152212142944
3/10 => loss is 0.16643740236759186
4/10 => loss is 0.1761135458946228
5/10 => loss is 0.16384334862232208
6/10 => loss is 0.255553275346756
7/10 => loss is 0.26424479484558105
8/10 => loss is 0.0937458798289299
9/10 => loss is 0.30804380774497986
10/10 => loss is 0.2702404856681824


In [132]:
model.eval()

with torch.no_grad():
    correct=0
    total=0

    for xb,yb in testloader:

        outputs=model(xb.unsqueeze(1))

        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

        total+=yb.size(0)

        correct+=(predicted==yb).sum().item()

        accuracy=correct/total*100

    print(accuracy)

85.83240899465564
